
# IPL Ball-by-Ball Data Analysis & Interactive Dashboard

**IBM Internship Project – Data Analytics**

This project analyzes the uploaded `Ball_by_Ball.csv` dataset using Python and creates an interactive Streamlit dashboard.

### Project objectives
- Clean and prepare IPL ball-by-ball data.
- Perform exploratory data analysis (EDA).
- Calculate batting, bowling, scoring and dismissal KPIs.
- Analyze runs by over and innings.
- Analyze extras and dismissal types.
- Compare teams and players using available ID fields.
- Build an interactive dashboard using Python, Pandas and Plotly.
- Generate a ready-to-run `app.py` Streamlit dashboard from this notebook.

> **Dataset note:** The supplied file contains player/team IDs rather than their names, so the analysis intentionally uses those IDs. This avoids inventing mappings that are not present in the uploaded dataset.


In [ ]:

# 1. Import libraries and load the dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("Ball_by_Ball.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:

# 2. Basic dataset inspection

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": [df[c].nunique(dropna=True) for c in df.columns]
}))

display(df.describe(include="all").T)


In [ ]:

# 3. Data cleaning and type conversion

# Standardize column names
df.columns = df.columns.str.strip()

# Strip whitespace from object columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Convert numeric columns
numeric_cols = [
    "Match_Id", "Innings_Id", "Over_Id", "Ball_Id",
    "Team_Batting_Id", "Team_Bowling_Id", "Striker_Id",
    "Striker_Batting_Position", "Non_Striker_Id", "Bowler_Id",
    "Batsman_Scored", "Extra_Runs", "Player_dissimal_Id", "Fielder_Id"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Missing/blank categorical values
categorical_cols = ["Extra_Type", "Dissimal_Type"]
for col in categorical_cols:
    df[col] = df[col].replace({"": "None", "nan": "None", " ": "None"}).fillna("None")

# Fill numeric values where blank cells represent zero
df["Batsman_Scored"] = df["Batsman_Scored"].fillna(0)
df["Extra_Runs"] = df["Extra_Runs"].fillna(0)

# Create useful analytical fields
df["Total_Runs"] = df["Batsman_Scored"] + df["Extra_Runs"]
df["Is_Dot_Ball"] = (df["Batsman_Scored"] == 0) & (df["Extra_Runs"] == 0)
df["Is_Boundary"] = df["Batsman_Scored"].isin([4, 6])
df["Is_Four"] = df["Batsman_Scored"] == 4
df["Is_Six"] = df["Batsman_Scored"] == 6
df["Is_Wicket"] = df["Dissimal_Type"].ne("None")

# Unique ball key
df["Ball_Key"] = (
    df["Match_Id"].astype("Int64").astype(str) + "_" +
    df["Innings_Id"].astype("Int64").astype(str) + "_" +
    df["Over_Id"].astype("Int64").astype(str) + "_" +
    df["Ball_Id"].astype("Int64").astype(str)
)

print("Cleaned shape:", df.shape)
display(df.head())


In [ ]:

# 4. Data quality checks

quality = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Unique matches",
        "Unique innings",
        "Unique batting teams",
        "Unique bowling teams",
        "Unique strikers",
        "Unique bowlers",
        "Total batsman runs",
        "Total extra runs",
        "Total runs",
        "Total wickets recorded",
        "Dot balls",
        "Fours",
        "Sixes"
    ],
    "Value": [
        len(df),
        df["Match_Id"].nunique(),
        df["Innings_Id"].nunique(),
        df["Team_Batting_Id"].nunique(),
        df["Team_Bowling_Id"].nunique(),
        df["Striker_Id"].nunique(),
        df["Bowler_Id"].nunique(),
        df["Batsman_Scored"].sum(),
        df["Extra_Runs"].sum(),
        df["Total_Runs"].sum(),
        df["Is_Wicket"].sum(),
        df["Is_Dot_Ball"].sum(),
        df["Is_Four"].sum(),
        df["Is_Six"].sum()
    ]
})

display(quality)


## 5. Executive KPIs

In [ ]:

total_runs = int(df["Total_Runs"].sum())
batting_runs = int(df["Batsman_Scored"].sum())
extra_runs = int(df["Extra_Runs"].sum())
wickets = int(df["Is_Wicket"].sum())
fours = int(df["Is_Four"].sum())
sixes = int(df["Is_Six"].sum())
dot_balls = int(df["Is_Dot_Ball"].sum())
matches = int(df["Match_Id"].nunique())

kpis = pd.DataFrame({
    "KPI": [
        "Matches", "Total Runs", "Batsman Runs", "Extra Runs",
        "Wickets", "Fours", "Sixes", "Dot Balls"
    ],
    "Value": [
        matches, total_runs, batting_runs, extra_runs,
        wickets, fours, sixes, dot_balls
    ]
})
display(kpis)


## 6. Team batting analysis

In [ ]:

team_batting = (
    df.groupby("Team_Batting_Id")
      .agg(
          Total_Runs=("Total_Runs", "sum"),
          Batting_Runs=("Batsman_Scored", "sum"),
          Extras=("Extra_Runs", "sum"),
          Balls=("Ball_Key", "nunique"),
          Fours=("Is_Four", "sum"),
          Sixes=("Is_Six", "sum")
      )
      .reset_index()
)

team_batting["Run_Rate_Per_Ball"] = team_batting["Total_Runs"] / team_batting["Balls"]
team_batting["Runs_Per_100_Balls"] = team_batting["Run_Rate_Per_Ball"] * 100
team_batting = team_batting.sort_values("Total_Runs", ascending=False)

display(team_batting.head(15))

fig = px.bar(
    team_batting.head(15),
    x="Team_Batting_Id",
    y="Total_Runs",
    text="Total_Runs",
    title="Top Batting Teams by Total Runs",
    labels={"Team_Batting_Id": "Team ID", "Total_Runs": "Total Runs"}
)
fig.update_traces(textposition="outside")
fig.show()


## 7. Player batting analysis

In [ ]:

player_batting = (
    df.groupby("Striker_Id")
      .agg(
          Runs=("Batsman_Scored", "sum"),
          Balls_Faced=("Ball_Key", "nunique"),
          Fours=("Is_Four", "sum"),
          Sixes=("Is_Six", "sum")
      )
      .reset_index()
)

player_batting["Strike_Rate"] = np.where(
    player_batting["Balls_Faced"] > 0,
    player_batting["Runs"] / player_batting["Balls_Faced"] * 100,
    0
)

player_batting = player_batting.sort_values(
    ["Runs", "Strike_Rate"], ascending=False
)

display(player_batting.head(20))

fig = px.bar(
    player_batting.head(15),
    x="Striker_Id",
    y="Runs",
    text="Runs",
    title="Top 15 Batters by Runs",
    labels={"Striker_Id": "Striker ID", "Runs": "Runs"}
)
fig.update_traces(textposition="outside")
fig.show()


## 8. Bowling analysis

In [ ]:

bowler_stats = (
    df.groupby("Bowler_Id")
      .agg(
          Balls=("Ball_Key", "nunique"),
          Runs_Conceded=("Total_Runs", "sum"),
          Wickets=("Is_Wicket", "sum"),
          Dot_Balls=("Is_Dot_Ball", "sum")
      )
      .reset_index()
)

bowler_stats["Economy_Per_Ball"] = np.where(
    bowler_stats["Balls"] > 0,
    bowler_stats["Runs_Conceded"] / bowler_stats["Balls"] * 6,
    0
)

bowler_stats["Dot_Ball_Percentage"] = (
    bowler_stats["Dot_Balls"] / bowler_stats["Balls"] * 100
)

# Minimum 60 balls gives a more meaningful comparison
qualified_bowlers = bowler_stats[bowler_stats["Balls"] >= 60].copy()

display(
    qualified_bowlers.sort_values(
        ["Wickets", "Economy_Per_Ball"], ascending=[False, True]
    ).head(20)
)

fig = px.bar(
    qualified_bowlers.sort_values("Wickets", ascending=False).head(15),
    x="Bowler_Id",
    y="Wickets",
    text="Wickets",
    title="Top Bowlers by Recorded Wickets",
    labels={"Bowler_Id": "Bowler ID", "Wickets": "Wickets"}
)
fig.update_traces(textposition="outside")
fig.show()


## 9. Runs by over

In [ ]:

over_analysis = (
    df.groupby("Over_Id")
      .agg(
          Runs=("Total_Runs", "sum"),
          Balls=("Ball_Key", "nunique"),
          Fours=("Is_Four", "sum"),
          Sixes=("Is_Six", "sum"),
          Wickets=("Is_Wicket", "sum")
      )
      .reset_index()
)

over_analysis["Runs_Per_Ball"] = over_analysis["Runs"] / over_analysis["Balls"]

display(over_analysis.head(20))

fig = px.line(
    over_analysis.sort_values("Over_Id"),
    x="Over_Id",
    y="Runs_Per_Ball",
    markers=True,
    title="Run Rate per Ball by Over Number",
    labels={"Over_Id": "Over", "Runs_Per_Ball": "Runs per Ball"}
)
fig.show()


## 10. Extras analysis

In [ ]:

extras = (
    df.groupby("Extra_Type")
      .agg(
          Extra_Runs=("Extra_Runs", "sum"),
          Balls=("Ball_Key", "nunique")
      )
      .reset_index()
)

extras = extras[extras["Extra_Type"] != "None"].sort_values(
    "Extra_Runs", ascending=False
)

display(extras)

fig = px.pie(
    extras,
    names="Extra_Type",
    values="Extra_Runs",
    title="Distribution of Extra Runs"
)
fig.show()


## 11. Dismissal analysis

In [ ]:

dismissals = (
    df[df["Dissimal_Type"] != "None"]
    .groupby("Dissimal_Type")
    .size()
    .reset_index(name="Wickets")
    .sort_values("Wickets", ascending=False)
)

display(dismissals)

fig = px.bar(
    dismissals,
    x="Dissimal_Type",
    y="Wickets",
    text="Wickets",
    title="Wickets by Dismissal Type",
    labels={"Dissimal_Type": "Dismissal Type", "Wickets": "Wickets"}
)
fig.update_traces(textposition="outside")
fig.show()


## 12. Match-level scoring analysis

In [ ]:

match_summary = (
    df.groupby(["Match_Id", "Innings_Id", "Team_Batting_Id"])
      .agg(
          Runs=("Total_Runs", "sum"),
          Batting_Runs=("Batsman_Scored", "sum"),
          Extras=("Extra_Runs", "sum"),
          Balls=("Ball_Key", "nunique"),
          Wickets=("Is_Wicket", "sum"),
          Fours=("Is_Four", "sum"),
          Sixes=("Is_Six", "sum")
      )
      .reset_index()
)

match_summary["Run_Rate"] = np.where(
    match_summary["Balls"] > 0,
    match_summary["Runs"] / match_summary["Balls"] * 6,
    0
)

display(match_summary.sort_values("Runs", ascending=False).head(20))

fig = px.histogram(
    match_summary,
    x="Runs",
    nbins=30,
    title="Distribution of Innings Scores",
    labels={"Runs": "Innings Runs"}
)
fig.show()


## 13. Correlation / relationship analysis

In [ ]:

corr_cols = [
    "Batsman_Scored", "Extra_Runs", "Total_Runs",
    "Over_Id", "Ball_Id", "Striker_Batting_Position"
]

corr = df[corr_cols].corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Correlation Matrix of Selected Numeric Features",
    aspect="auto"
)
fig.show()


## 14. Business-style insights

In [ ]:

top_team = team_batting.iloc[0]
top_batter = player_batting.iloc[0]
top_bowler = qualified_bowlers.sort_values(
    ["Wickets", "Economy_Per_Ball"], ascending=[False, True]
).iloc[0] if len(qualified_bowlers) else None

print("KEY INSIGHTS")
print("-" * 70)
print(f"1. The dataset contains {matches:,} matches and {len(df):,} ball records.")
print(f"2. Total runs recorded: {total_runs:,}.")
print(f"3. Team ID {int(top_team['Team_Batting_Id'])} has the highest aggregate batting runs: {int(top_team['Total_Runs']):,}.")
print(f"4. Striker ID {int(top_batter['Striker_Id'])} has the highest aggregate runs: {int(top_batter['Runs']):,}.")
if top_bowler is not None:
    print(f"5. Bowler ID {int(top_bowler['Bowler_Id'])} has the highest recorded wickets among bowlers with at least 60 balls: {int(top_bowler['Wickets']):,}.")
print(f"6. Boundary count: {fours:,} fours and {sixes:,} sixes.")
print(f"7. Dot balls: {dot_balls:,}, representing {dot_balls/len(df)*100:.2f}% of all records.")
print(f"8. Extra runs: {extra_runs:,}, representing {extra_runs/total_runs*100:.2f}% of total runs.")



# 15. Interactive Streamlit Dashboard

The following cell contains the **complete dashboard code**. It creates a file named `app.py`.

### Run the dashboard
From the same folder containing `Ball_by_Ball.csv` and `app.py`:

```bash
pip install pandas numpy plotly streamlit
streamlit run app.py
```

The dashboard includes:
- KPI cards
- Team filter
- Innings filter
- Batting analysis
- Bowling analysis
- Runs-by-over chart
- Extras chart
- Dismissal chart
- Match/innings score distribution
- Detailed filtered data table


In [ ]:
APP_CODE = '\nimport streamlit as st\nimport pandas as pd\nimport numpy as np\nimport plotly.express as px\nfrom pathlib import Path\n\nst.set_page_config(\n    page_title="IPL Ball-by-Ball Analytics",\n    page_icon="🏏",\n    layout="wide"\n)\n\nst.title("🏏 IPL Ball-by-Ball Analytics Dashboard")\nst.caption("IBM Internship Data Analytics Project | Python + Pandas + Plotly + Streamlit")\n\n@st.cache_data\ndef load_data():\n    path = Path("Ball_by_Ball.csv")\n    data = pd.read_csv(path)\n\n    data.columns = data.columns.str.strip()\n\n    for col in data.select_dtypes(include="object").columns:\n        data[col] = data[col].astype(str).str.strip()\n\n    numeric_cols = [\n        "Match_Id", "Innings_Id", "Over_Id", "Ball_Id",\n        "Team_Batting_Id", "Team_Bowling_Id", "Striker_Id",\n        "Striker_Batting_Position", "Non_Striker_Id", "Bowler_Id",\n        "Batsman_Scored", "Extra_Runs", "Player_dissimal_Id", "Fielder_Id"\n    ]\n\n    for col in numeric_cols:\n        data[col] = pd.to_numeric(data[col], errors="coerce")\n\n    data["Batsman_Scored"] = data["Batsman_Scored"].fillna(0)\n    data["Extra_Runs"] = data["Extra_Runs"].fillna(0)\n\n    for col in ["Extra_Type", "Dissimal_Type"]:\n        data[col] = data[col].replace(\n            {"": "None", "nan": "None", " ": "None"}\n        ).fillna("None")\n\n    data["Total_Runs"] = data["Batsman_Scored"] + data["Extra_Runs"]\n    data["Is_Dot_Ball"] = (\n        (data["Batsman_Scored"] == 0) &\n        (data["Extra_Runs"] == 0)\n    )\n    data["Is_Four"] = data["Batsman_Scored"] == 4\n    data["Is_Six"] = data["Batsman_Scored"] == 6\n    data["Is_Wicket"] = data["Dissimal_Type"].ne("None")\n\n    data["Ball_Key"] = (\n        data["Match_Id"].astype("Int64").astype(str) + "_" +\n        data["Innings_Id"].astype("Int64").astype(str) + "_" +\n        data["Over_Id"].astype("Int64").astype(str) + "_" +\n        data["Ball_Id"].astype("Int64").astype(str)\n    )\n\n    return data\n\ndf = load_data()\n\n# -------------------- SIDEBAR FILTERS --------------------\nst.sidebar.header("Dashboard Filters")\n\nteam_options = sorted(df["Team_Batting_Id"].dropna().unique().tolist())\nteam_filter = st.sidebar.multiselect(\n    "Batting Team ID",\n    options=team_options,\n    default=team_options\n)\n\ninnings_options = sorted(df["Innings_Id"].dropna().unique().tolist())\ninnings_filter = st.sidebar.multiselect(\n    "Innings ID",\n    options=innings_options,\n    default=innings_options\n)\n\nfiltered = df[\n    df["Team_Batting_Id"].isin(team_filter) &\n    df["Innings_Id"].isin(innings_filter)\n].copy()\n\n# -------------------- KPIs --------------------\ntotal_runs = int(filtered["Total_Runs"].sum())\nmatches = int(filtered["Match_Id"].nunique())\nwickets = int(filtered["Is_Wicket"].sum())\nfours = int(filtered["Is_Four"].sum())\nsixes = int(filtered["Is_Six"].sum())\ndot_balls = int(filtered["Is_Dot_Ball"].sum())\n\nc1, c2, c3, c4, c5, c6 = st.columns(6)\n\nc1.metric("Matches", f"{matches:,}")\nc2.metric("Total Runs", f"{total_runs:,}")\nc3.metric("Wickets", f"{wickets:,}")\nc4.metric("Fours", f"{fours:,}")\nc5.metric("Sixes", f"{sixes:,}")\nc6.metric("Dot Balls", f"{dot_balls:,}")\n\nst.divider()\n\nif filtered.empty:\n    st.warning("No data matches the selected filters.")\n    st.stop()\n\n# -------------------- TEAM ANALYSIS --------------------\nleft, right = st.columns(2)\n\nteam_stats = (\n    filtered.groupby("Team_Batting_Id")\n    .agg(\n        Runs=("Total_Runs", "sum"),\n        Balls=("Ball_Key", "nunique"),\n        Fours=("Is_Four", "sum"),\n        Sixes=("Is_Six", "sum")\n    )\n    .reset_index()\n)\n\nteam_stats["Run_Rate"] = np.where(\n    team_stats["Balls"] > 0,\n    team_stats["Runs"] / team_stats["Balls"] * 6,\n    0\n)\n\nwith left:\n    fig = px.bar(\n        team_stats.sort_values("Runs", ascending=False),\n        x="Team_Batting_Id",\n        y="Runs",\n        text="Runs",\n        title="Team Runs"\n    )\n    fig.update_traces(textposition="outside")\n    st.plotly_chart(fig, use_container_width=True)\n\nwith right:\n    fig = px.bar(\n        team_stats.sort_values("Run_Rate", ascending=False),\n        x="Team_Batting_Id",\n        y="Run_Rate",\n        text="Run_Rate",\n        title="Team Run Rate"\n    )\n    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")\n    st.plotly_chart(fig, use_container_width=True)\n\n# -------------------- PLAYER ANALYSIS --------------------\nst.subheader("🏏 Batting Performance")\n\nplayer_stats = (\n    filtered.groupby("Striker_Id")\n    .agg(\n        Runs=("Batsman_Scored", "sum"),\n        Balls=("Ball_Key", "nunique"),\n        Fours=("Is_Four", "sum"),\n        Sixes=("Is_Six", "sum")\n    )\n    .reset_index()\n)\n\nplayer_stats["Strike_Rate"] = np.where(\n    player_stats["Balls"] > 0,\n    player_stats["Runs"] / player_stats["Balls"] * 100,\n    0\n)\n\ntop_players = player_stats.sort_values(\n    ["Runs", "Strike_Rate"], ascending=False\n).head(15)\n\nfig = px.bar(\n    top_players,\n    x="Striker_Id",\n    y="Runs",\n    text="Runs",\n    hover_data=["Balls", "Fours", "Sixes", "Strike_Rate"],\n    title="Top 15 Batters by Runs"\n)\nfig.update_traces(textposition="outside")\nst.plotly_chart(fig, use_container_width=True)\n\n# -------------------- BOWLING --------------------\nst.subheader("🎯 Bowling Performance")\n\nbowler_stats = (\n    filtered.groupby("Bowler_Id")\n    .agg(\n        Balls=("Ball_Key", "nunique"),\n        Runs_Conceded=("Total_Runs", "sum"),\n        Wickets=("Is_Wicket", "sum"),\n        Dot_Balls=("Is_Dot_Ball", "sum")\n    )\n    .reset_index()\n)\n\nbowler_stats["Economy"] = np.where(\n    bowler_stats["Balls"] > 0,\n    bowler_stats["Runs_Conceded"] / bowler_stats["Balls"] * 6,\n    0\n)\n\nbowler_stats["Dot_Ball_Percentage"] = np.where(\n    bowler_stats["Balls"] > 0,\n    bowler_stats["Dot_Balls"] / bowler_stats["Balls"] * 100,\n    0\n)\n\nqualified = bowler_stats[bowler_stats["Balls"] >= 60].copy()\n\nif not qualified.empty:\n    left, right = st.columns(2)\n\n    with left:\n        fig = px.bar(\n            qualified.sort_values("Wickets", ascending=False).head(15),\n            x="Bowler_Id",\n            y="Wickets",\n            text="Wickets",\n            title="Top Bowlers by Wickets"\n        )\n        fig.update_traces(textposition="outside")\n        st.plotly_chart(fig, use_container_width=True)\n\n    with right:\n        economy_chart = qualified.sort_values("Economy").head(15)\n        fig = px.bar(\n            economy_chart,\n            x="Bowler_Id",\n            y="Economy",\n            text="Economy",\n            title="Best Economy Among Bowlers with 60+ Balls"\n        )\n        fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")\n        st.plotly_chart(fig, use_container_width=True)\n\n# -------------------- OVER ANALYSIS --------------------\nst.subheader("📈 Scoring by Over")\n\nover_stats = (\n    filtered.groupby("Over_Id")\n    .agg(\n        Runs=("Total_Runs", "sum"),\n        Balls=("Ball_Key", "nunique"),\n        Wickets=("Is_Wicket", "sum")\n    )\n    .reset_index()\n)\n\nover_stats["Run_Rate"] = np.where(\n    over_stats["Balls"] > 0,\n    over_stats["Runs"] / over_stats["Balls"] * 6,\n    0\n)\n\nfig = px.line(\n    over_stats.sort_values("Over_Id"),\n    x="Over_Id",\n    y="Run_Rate",\n    markers=True,\n    title="Run Rate by Over Number"\n)\nst.plotly_chart(fig, use_container_width=True)\n\n# -------------------- EXTRAS + DISMISSALS --------------------\nleft, right = st.columns(2)\n\nwith left:\n    extras = (\n        filtered[filtered["Extra_Type"] != "None"]\n        .groupby("Extra_Type")["Extra_Runs"]\n        .sum()\n        .reset_index()\n        .sort_values("Extra_Runs", ascending=False)\n    )\n\n    fig = px.pie(\n        extras,\n        names="Extra_Type",\n        values="Extra_Runs",\n        title="Extra Runs Distribution"\n    )\n    st.plotly_chart(fig, use_container_width=True)\n\nwith right:\n    dismissals = (\n        filtered[filtered["Dissimal_Type"] != "None"]\n        .groupby("Dissimal_Type")\n        .size()\n        .reset_index(name="Wickets")\n        .sort_values("Wickets", ascending=False)\n    )\n\n    fig = px.bar(\n        dismissals,\n        x="Dissimal_Type",\n        y="Wickets",\n        text="Wickets",\n        title="Dismissal Types"\n    )\n    fig.update_traces(textposition="outside")\n    st.plotly_chart(fig, use_container_width=True)\n\n# -------------------- SCORE DISTRIBUTION --------------------\nst.subheader("📊 Innings Score Distribution")\n\ninnings_scores = (\n    filtered.groupby(["Match_Id", "Innings_Id", "Team_Batting_Id"])\n    .agg(Runs=("Total_Runs", "sum"))\n    .reset_index()\n)\n\nfig = px.histogram(\n    innings_scores,\n    x="Runs",\n    nbins=30,\n    title="Distribution of Innings Scores"\n)\nst.plotly_chart(fig, use_container_width=True)\n\n# -------------------- DATA TABLE --------------------\nst.subheader("🔎 Filtered Ball-by-Ball Data")\n\ndisplay_cols = [\n    "Match_Id", "Innings_Id", "Over_Id", "Ball_Id",\n    "Team_Batting_Id", "Team_Bowling_Id", "Striker_Id",\n    "Bowler_Id", "Batsman_Scored", "Extra_Type",\n    "Extra_Runs", "Total_Runs", "Dissimal_Type"\n]\n\nst.dataframe(\n    filtered[display_cols].sort_values(\n        ["Match_Id", "Innings_Id", "Over_Id", "Ball_Id"]\n    ),\n    use_container_width=True,\n    height=400\n)\n\n# -------------------- DOWNLOAD --------------------\ncsv = filtered.to_csv(index=False).encode("utf-8")\n\nst.download_button(\n    label="⬇️ Download Filtered Data",\n    data=csv,\n    file_name="filtered_ball_by_ball.csv",\n    mime="text/csv"\n)\n\nst.caption(\n    "Project created for data analytics practice and IBM internship submission. "\n    "Player/team names are not inferred because the uploaded dataset contains IDs only."\n)\n'

In [ ]:
# 16. Generate app.py from the dashboard code stored below

from pathlib import Path

app_path = Path('app.py')
app_path.write_text(APP_CODE, encoding='utf-8')
print(f'Created: {app_path.resolve()}')
print('Run with: streamlit run app.py')

In [ ]:

# 16. Create the Streamlit app.py file from this notebook

from pathlib import Path

app_path = Path("app.py")
app_path.write_text('\nimport streamlit as st\nimport pandas as pd\nimport numpy as np\nimport plotly.express as px\nfrom pathlib import Path\n\nst.set_page_config(\n    page_title="IPL Ball-by-Ball Analytics",\n    page_icon="🏏",\n    layout="wide"\n)\n\nst.title("🏏 IPL Ball-by-Ball Analytics Dashboard")\nst.caption("IBM Internship Data Analytics Project | Python + Pandas + Plotly + Streamlit")\n\n@st.cache_data\ndef load_data():\n    path = Path("Ball_by_Ball.csv")\n    data = pd.read_csv(path)\n\n    data.columns = data.columns.str.strip()\n\n    for col in data.select_dtypes(include="object").columns:\n        data[col] = data[col].astype(str).str.strip()\n\n    numeric_cols = [\n        "Match_Id", "Innings_Id", "Over_Id", "Ball_Id",\n        "Team_Batting_Id", "Team_Bowling_Id", "Striker_Id",\n        "Striker_Batting_Position", "Non_Striker_Id", "Bowler_Id",\n        "Batsman_Scored", "Extra_Runs", "Player_dissimal_Id", "Fielder_Id"\n    ]\n\n    for col in numeric_cols:\n        data[col] = pd.to_numeric(data[col], errors="coerce")\n\n    data["Batsman_Scored"] = data["Batsman_Scored"].fillna(0)\n    data["Extra_Runs"] = data["Extra_Runs"].fillna(0)\n\n    for col in ["Extra_Type", "Dissimal_Type"]:\n        data[col] = data[col].replace(\n            {"": "None", "nan": "None", " ": "None"}\n        ).fillna("None")\n\n    data["Total_Runs"] = data["Batsman_Scored"] + data["Extra_Runs"]\n    data["Is_Dot_Ball"] = (\n        (data["Batsman_Scored"] == 0) &\n        (data["Extra_Runs"] == 0)\n    )\n    data["Is_Four"] = data["Batsman_Scored"] == 4\n    data["Is_Six"] = data["Batsman_Scored"] == 6\n    data["Is_Wicket"] = data["Dissimal_Type"].ne("None")\n\n    data["Ball_Key"] = (\n        data["Match_Id"].astype("Int64").astype(str) + "_" +\n        data["Innings_Id"].astype("Int64").astype(str) + "_" +\n        data["Over_Id"].astype("Int64").astype(str) + "_" +\n        data["Ball_Id"].astype("Int64").astype(str)\n    )\n\n    return data\n\ndf = load_data()\n\n# -------------------- SIDEBAR FILTERS --------------------\nst.sidebar.header("Dashboard Filters")\n\nteam_options = sorted(df["Team_Batting_Id"].dropna().unique().tolist())\nteam_filter = st.sidebar.multiselect(\n    "Batting Team ID",\n    options=team_options,\n    default=team_options\n)\n\ninnings_options = sorted(df["Innings_Id"].dropna().unique().tolist())\ninnings_filter = st.sidebar.multiselect(\n    "Innings ID",\n    options=innings_options,\n    default=innings_options\n)\n\nfiltered = df[\n    df["Team_Batting_Id"].isin(team_filter) &\n    df["Innings_Id"].isin(innings_filter)\n].copy()\n\n# -------------------- KPIs --------------------\ntotal_runs = int(filtered["Total_Runs"].sum())\nmatches = int(filtered["Match_Id"].nunique())\nwickets = int(filtered["Is_Wicket"].sum())\nfours = int(filtered["Is_Four"].sum())\nsixes = int(filtered["Is_Six"].sum())\ndot_balls = int(filtered["Is_Dot_Ball"].sum())\n\nc1, c2, c3, c4, c5, c6 = st.columns(6)\n\nc1.metric("Matches", f"{matches:,}")\nc2.metric("Total Runs", f"{total_runs:,}")\nc3.metric("Wickets", f"{wickets:,}")\nc4.metric("Fours", f"{fours:,}")\nc5.metric("Sixes", f"{sixes:,}")\nc6.metric("Dot Balls", f"{dot_balls:,}")\n\nst.divider()\n\nif filtered.empty:\n    st.warning("No data matches the selected filters.")\n    st.stop()\n\n# -------------------- TEAM ANALYSIS --------------------\nleft, right = st.columns(2)\n\nteam_stats = (\n    filtered.groupby("Team_Batting_Id")\n    .agg(\n        Runs=("Total_Runs", "sum"),\n        Balls=("Ball_Key", "nunique"),\n        Fours=("Is_Four", "sum"),\n        Sixes=("Is_Six", "sum")\n    )\n    .reset_index()\n)\n\nteam_stats["Run_Rate"] = np.where(\n    team_stats["Balls"] > 0,\n    team_stats["Runs"] / team_stats["Balls"] * 6,\n    0\n)\n\nwith left:\n    fig = px.bar(\n        team_stats.sort_values("Runs", ascending=False),\n        x="Team_Batting_Id",\n        y="Runs",\n        text="Runs",\n        title="Team Runs"\n    )\n    fig.update_traces(textposition="outside")\n    st.plotly_chart(fig, use_container_width=True)\n\nwith right:\n    fig = px.bar(\n        team_stats.sort_values("Run_Rate", ascending=False),\n        x="Team_Batting_Id",\n        y="Run_Rate",\n        text="Run_Rate",\n        title="Team Run Rate"\n    )\n    fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")\n    st.plotly_chart(fig, use_container_width=True)\n\n# -------------------- PLAYER ANALYSIS --------------------\nst.subheader("🏏 Batting Performance")\n\nplayer_stats = (\n    filtered.groupby("Striker_Id")\n    .agg(\n        Runs=("Batsman_Scored", "sum"),\n        Balls=("Ball_Key", "nunique"),\n        Fours=("Is_Four", "sum"),\n        Sixes=("Is_Six", "sum")\n    )\n    .reset_index()\n)\n\nplayer_stats["Strike_Rate"] = np.where(\n    player_stats["Balls"] > 0,\n    player_stats["Runs"] / player_stats["Balls"] * 100,\n    0\n)\n\ntop_players = player_stats.sort_values(\n    ["Runs", "Strike_Rate"], ascending=False\n).head(15)\n\nfig = px.bar(\n    top_players,\n    x="Striker_Id",\n    y="Runs",\n    text="Runs",\n    hover_data=["Balls", "Fours", "Sixes", "Strike_Rate"],\n    title="Top 15 Batters by Runs"\n)\nfig.update_traces(textposition="outside")\nst.plotly_chart(fig, use_container_width=True)\n\n# -------------------- BOWLING --------------------\nst.subheader("🎯 Bowling Performance")\n\nbowler_stats = (\n    filtered.groupby("Bowler_Id")\n    .agg(\n        Balls=("Ball_Key", "nunique"),\n        Runs_Conceded=("Total_Runs", "sum"),\n        Wickets=("Is_Wicket", "sum"),\n        Dot_Balls=("Is_Dot_Ball", "sum")\n    )\n    .reset_index()\n)\n\nbowler_stats["Economy"] = np.where(\n    bowler_stats["Balls"] > 0,\n    bowler_stats["Runs_Conceded"] / bowler_stats["Balls"] * 6,\n    0\n)\n\nbowler_stats["Dot_Ball_Percentage"] = np.where(\n    bowler_stats["Balls"] > 0,\n    bowler_stats["Dot_Balls"] / bowler_stats["Balls"] * 100,\n    0\n)\n\nqualified = bowler_stats[bowler_stats["Balls"] >= 60].copy()\n\nif not qualified.empty:\n    left, right = st.columns(2)\n\n    with left:\n        fig = px.bar(\n            qualified.sort_values("Wickets", ascending=False).head(15),\n            x="Bowler_Id",\n            y="Wickets",\n            text="Wickets",\n            title="Top Bowlers by Wickets"\n        )\n        fig.update_traces(textposition="outside")\n        st.plotly_chart(fig, use_container_width=True)\n\n    with right:\n        economy_chart = qualified.sort_values("Economy").head(15)\n        fig = px.bar(\n            economy_chart,\n            x="Bowler_Id",\n            y="Economy",\n            text="Economy",\n            title="Best Economy Among Bowlers with 60+ Balls"\n        )\n        fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")\n        st.plotly_chart(fig, use_container_width=True)\n\n# -------------------- OVER ANALYSIS --------------------\nst.subheader("📈 Scoring by Over")\n\nover_stats = (\n    filtered.groupby("Over_Id")\n    .agg(\n        Runs=("Total_Runs", "sum"),\n        Balls=("Ball_Key", "nunique"),\n        Wickets=("Is_Wicket", "sum")\n    )\n    .reset_index()\n)\n\nover_stats["Run_Rate"] = np.where(\n    over_stats["Balls"] > 0,\n    over_stats["Runs"] / over_stats["Balls"] * 6,\n    0\n)\n\nfig = px.line(\n    over_stats.sort_values("Over_Id"),\n    x="Over_Id",\n    y="Run_Rate",\n    markers=True,\n    title="Run Rate by Over Number"\n)\nst.plotly_chart(fig, use_container_width=True)\n\n# -------------------- EXTRAS + DISMISSALS --------------------\nleft, right = st.columns(2)\n\nwith left:\n    extras = (\n        filtered[filtered["Extra_Type"] != "None"]\n        .groupby("Extra_Type")["Extra_Runs"]\n        .sum()\n        .reset_index()\n        .sort_values("Extra_Runs", ascending=False)\n    )\n\n    fig = px.pie(\n        extras,\n        names="Extra_Type",\n        values="Extra_Runs",\n        title="Extra Runs Distribution"\n    )\n    st.plotly_chart(fig, use_container_width=True)\n\nwith right:\n    dismissals = (\n        filtered[filtered["Dissimal_Type"] != "None"]\n        .groupby("Dissimal_Type")\n        .size()\n        .reset_index(name="Wickets")\n        .sort_values("Wickets", ascending=False)\n    )\n\n    fig = px.bar(\n        dismissals,\n        x="Dissimal_Type",\n        y="Wickets",\n        text="Wickets",\n        title="Dismissal Types"\n    )\n    fig.update_traces(textposition="outside")\n    st.plotly_chart(fig, use_container_width=True)\n\n# -------------------- SCORE DISTRIBUTION --------------------\nst.subheader("📊 Innings Score Distribution")\n\ninnings_scores = (\n    filtered.groupby(["Match_Id", "Innings_Id", "Team_Batting_Id"])\n    .agg(Runs=("Total_Runs", "sum"))\n    .reset_index()\n)\n\nfig = px.histogram(\n    innings_scores,\n    x="Runs",\n    nbins=30,\n    title="Distribution of Innings Scores"\n)\nst.plotly_chart(fig, use_container_width=True)\n\n# -------------------- DATA TABLE --------------------\nst.subheader("🔎 Filtered Ball-by-Ball Data")\n\ndisplay_cols = [\n    "Match_Id", "Innings_Id", "Over_Id", "Ball_Id",\n    "Team_Batting_Id", "Team_Bowling_Id", "Striker_Id",\n    "Bowler_Id", "Batsman_Scored", "Extra_Type",\n    "Extra_Runs", "Total_Runs", "Dissimal_Type"\n]\n\nst.dataframe(\n    filtered[display_cols].sort_values(\n        ["Match_Id", "Innings_Id", "Over_Id", "Ball_Id"]\n    ),\n    use_container_width=True,\n    height=400\n)\n\n# -------------------- DOWNLOAD --------------------\ncsv = filtered.to_csv(index=False).encode("utf-8")\n\nst.download_button(\n    label="⬇️ Download Filtered Data",\n    data=csv,\n    file_name="filtered_ball_by_ball.csv",\n    mime="text/csv"\n)\n\nst.caption(\n    "Project created for data analytics practice and IBM internship submission. "\n    "Player/team names are not inferred because the uploaded dataset contains IDs only."\n)\n', encoding="utf-8")

print(f"Created: {app_path.resolve()}")
print("Run with: streamlit run app.py")



## 17. Suggested project submission structure

```text
IPL_Ball_by_Ball_IBM_Project/
│
├── Ball_by_Ball.csv
├── IPL_Ball_by_Ball_Analysis.ipynb
├── app.py
└── README.md
```

### Recommended project title
**IPL Ball-by-Ball Data Analysis and Interactive Dashboard using Python**

### Tools used
- Python
- Pandas
- NumPy
- Matplotlib
- Plotly
- Streamlit
- Jupyter Notebook

### Main analytical areas
- Data cleaning and preprocessing
- Exploratory Data Analysis
- KPI analysis
- Team performance
- Batter performance
- Bowler performance
- Run-rate analysis
- Extras analysis
- Dismissal analysis
- Interactive dashboard
